In [1]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
import pyodbc
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────
# 1. CONEXIÓN AL DATAMART
# ─────────────────────────────────────────────────────────────
def get_connection():
    conn = pyodbc.connect(
        'DRIVER={ODBC Driver 17 for SQL Server};'
        'SERVER=LAPTOP-RSS8LMV8;'
        'DATABASE=proyecto_destino;'          # FIX: era proyecto_destino2
        'UID=sa;'
        'PWD=@DDJJrrvv1303',
        timeout=30
    )
    return conn


# ─────────────────────────────────────────────────────────────
# 2. CARGA DE DATOS
# ─────────────────────────────────────────────────────────────
def load_data():
    print("Conectando a proyecto_destino...")
    conn = get_connection()
    print("Conexión exitosa")

    query = """
        SELECT
            fa.id,
            fa.duracion_minutos,
            fa.distance,
            fa.temperature,
            fa.humidity,
            fa.visibility,
            fa.wind_speed,
            fa.precipitation,

            -- DimSeverity
            ds.severity,

            -- DimFecha
            df.start_time,
            df.anio,
            df.mes,
            df.dia,
            df.hora,
            df.dia_semana,
            df.es_fin_semana,

            -- DimLugar
            dl.street,
            dl.city,
            dl.county,
            dl.state,
            dl.zipcode,
            dl.timezone,

            -- DimClima
            dc.weather_condition,

            -- DimCivilTwilight
            dct.civil_twilight,

            -- Infraestructura: son VARCHAR('True'/'False'), se leen tal cual
            dcr.crossing,
            dj.junction,
            dst2.station,
            dsp.stop,
            dtr.traffic_signal

        FROM FactAccidente fa
        JOIN DimSeverity      ds   ON fa.severityID      = ds.id
        JOIN DimFecha         df   ON fa.fechaID         = df.id
        JOIN DimLugar         dl   ON fa.lugarID         = dl.id
        JOIN DimClima         dc   ON fa.climaID         = dc.id
        JOIN DimCivilTwilight dct  ON fa.civilTwilightID = dct.id
        JOIN DimCrossing      dcr  ON fa.crossingID      = dcr.id
        JOIN DimJunction      dj   ON fa.junctionID      = dj.id
        JOIN DimStation       dst2 ON fa.stationID       = dst2.id
        JOIN DimStop          dsp  ON fa.stopID          = dsp.id
        JOIN DimTraffic       dtr  ON fa.trafficID       = dtr.id
    """

    df = pd.read_sql(query, conn)
    conn.close()

    df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
    df['severity']   = pd.to_numeric(df['severity'], errors='coerce').astype('Int64')

    # FIX: las columnas de infraestructura son VARCHAR 'True'/'False'
    # Convertir a booleano numérico para poder filtrar con == 1
    for col in ['crossing', 'junction', 'station', 'stop', 'traffic_signal']:
        df[col] = df[col].map({'True': 1, 'False': 0}).fillna(0).astype(int)

    print(f"Datos cargados: {len(df):,} registros")
    return df


# ─────────────────────────────────────────────────────────────
# 3. CARGA INICIAL
# ─────────────────────────────────────────────────────────────
print("Iniciando carga de datos...")
df_main = load_data()

ESTADOS     = sorted(df_main['state'].dropna().unique())
CLIMAS      = sorted(df_main['weather_condition'].dropna().unique())
ANIOS       = sorted(df_main['anio'].dropna().unique())
SEVERIDADES = sorted(df_main['severity'].dropna().unique())

COLOR_SEV = {1: '#2ecc71', 2: '#f39c12', 3: '#e67e22', 4: '#e74c3c'}

# FIX: mapa de días en inglés (como los guarda DimFecha con dt.day_name())
DIAS_MAP_EN = {
    'Monday':    'Lun',
    'Tuesday':   'Mar',
    'Wednesday': 'Mié',
    'Thursday':  'Jue',
    'Friday':    'Vie',
    'Saturday':  'Sáb',
    'Sunday':    'Dom',
}
DIAS_ORDER = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']

# ─────────────────────────────────────────────────────────────
# 4. APP DASH
# ─────────────────────────────────────────────────────────────
app = dash.Dash(__name__, suppress_callback_exceptions=True)
app.title = "BI Accidentes USA"

CARD = {
    'backgroundColor': 'white',
    'borderRadius': '10px',
    'padding': '20px',
    'boxShadow': '0 2px 6px rgba(0,0,0,0.10)',
    'textAlign': 'center',
}
PANEL = {
    'backgroundColor': '#f0f4f8',
    'borderRadius': '10px',
    'padding': '15px',
    'marginBottom': '20px',
}
GRAPH_CARD = {
    'backgroundColor': 'white',
    'borderRadius': '10px',
    'padding': '8px',
    'boxShadow': '0 2px 6px rgba(0,0,0,0.08)',
}

app.layout = html.Div(
    style={'fontFamily': 'Segoe UI, Arial, sans-serif',
           'backgroundColor': '#eef1f5', 'padding': '20px'},
    children=[

    # ── ENCABEZADO ──────────────────────────────────────────
    html.Div([
        html.H1("Dashboard — Accidentes de Tránsito en EE.UU.",
                style={'textAlign': 'center', 'color': '#2c3e50', 'marginBottom': '4px'}),
        html.P("Análisis multidimensional · Datamart BI · US-Accidents (2016–2023)",
               style={'textAlign': 'center', 'color': '#7f8c8d', 'marginBottom': '4px'}),
        html.P("Integrantes: Diego Saldaña · Diego Rodríguez · Sebastián Vinces · Mario Auqui",
               style={'textAlign': 'center', 'color': '#95a5a6', 'fontSize': '12px'}),
    ], style={'marginBottom': '20px'}),

    # ── FILTROS ─────────────────────────────────────────────
    html.Div([
        html.Div([
            html.Label("Estado:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='filter-state',
                options=[{'label': 'Todos', 'value': 'all'}] +
                        [{'label': s, 'value': s} for s in ESTADOS],
                value='all', clearable=False
            )
        ], style={'width': '18%', 'display': 'inline-block', 'marginRight': '1%'}),

        html.Div([
            html.Label("Severidad:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='filter-severity',
                options=[{'label': 'Todas', 'value': 'all'}] +
                        [{'label': f'Severidad {s}', 'value': s} for s in SEVERIDADES],
                value='all', clearable=False
            )
        ], style={'width': '18%', 'display': 'inline-block', 'marginRight': '1%'}),

        html.Div([
            html.Label("Condición climática:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='filter-clima',
                options=[{'label': 'Todas', 'value': 'all'}] +
                        [{'label': c, 'value': c} for c in CLIMAS],
                value='all', clearable=False
            )
        ], style={'width': '22%', 'display': 'inline-block', 'marginRight': '1%'}),

        html.Div([
            html.Label("Condición de luz:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='filter-twilight',
                options=[
                    {'label': 'Día y Noche', 'value': 'all'},
                    {'label': 'Día',         'value': 'Day'},
                    {'label': 'Noche',       'value': 'Night'},
                ],
                value='all', clearable=False
            )
        ], style={'width': '18%', 'display': 'inline-block', 'marginRight': '1%'}),

        html.Div([
            html.Label("Año:", style={'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='filter-anio',
                options=[{'label': 'Todos', 'value': 'all'}] +
                        [{'label': str(a), 'value': a} for a in ANIOS],
                value='all', clearable=False
            )
        ], style={'width': '18%', 'display': 'inline-block'}),
    ], style=PANEL),

    # ── KPI CARDS ───────────────────────────────────────────
    html.Div(id='kpi-cards',
             style={'display': 'grid',
                    'gridTemplateColumns': 'repeat(5, 1fr)',
                    'gap': '15px',
                    'marginBottom': '20px'}),

    # ── FILA 1 ──────────────────────────────────────────────
    html.Div([
        html.Div([dcc.Graph(id='chart-heatmap-hora-sev')],
                 style={**GRAPH_CARD, 'width': '58%', 'display': 'inline-block'}),
        html.Div([dcc.Graph(id='chart-dia-luz')],
                 style={**GRAPH_CARD, 'width': '39%', 'float': 'right', 'display': 'inline-block'}),
    ], style={'marginBottom': '20px'}),

    # ── FILA 2 ──────────────────────────────────────────────
    html.Div([
        html.Div([dcc.Graph(id='chart-top-states')],
                 style={**GRAPH_CARD, 'width': '48%', 'display': 'inline-block'}),
        html.Div([dcc.Graph(id='chart-top-cities')],
                 style={**GRAPH_CARD, 'width': '48%', 'float': 'right', 'display': 'inline-block'}),
    ], style={'marginBottom': '20px'}),

    # ── FILA 3 ──────────────────────────────────────────────
    html.Div([
        html.Div([dcc.Graph(id='chart-clima-count')],
                 style={**GRAPH_CARD, 'width': '48%', 'display': 'inline-block'}),
        html.Div([dcc.Graph(id='chart-clima-sev-prom')],
                 style={**GRAPH_CARD, 'width': '48%', 'float': 'right', 'display': 'inline-block'}),
    ], style={'marginBottom': '20px'}),

    # ── FILA 4 ──────────────────────────────────────────────
    html.Div([dcc.Graph(id='chart-infraestructura')],
             style={**GRAPH_CARD, 'marginBottom': '20px'}),

    # ── FILA 5 ──────────────────────────────────────────────
    html.Div([
        html.Div([dcc.Graph(id='chart-dur-severidad')],
                 style={**GRAPH_CARD, 'width': '32%', 'display': 'inline-block'}),
        html.Div([dcc.Graph(id='chart-dur-clima')],
                 style={**GRAPH_CARD, 'width': '32%', 'display': 'inline-block', 'marginLeft': '1%'}),
        html.Div([dcc.Graph(id='chart-dur-estado')],
                 style={**GRAPH_CARD, 'width': '32%', 'float': 'right', 'display': 'inline-block'}),
    ], style={'marginBottom': '20px'}),

    # ── PIE ─────────────────────────────────────────────────
    html.Div([dcc.Graph(id='chart-severity-pie')],
             style={**GRAPH_CARD, 'marginBottom': '20px'}),

    # ── NOTA ────────────────────────────────────────────────
    html.Div([
        html.P("Todos los datos provienen de proyecto_destino (SQL Server). "
               "Los filtros superiores se aplican en tiempo real a todos los gráficos y KPIs.",
               style={'color': '#7f8c8d', 'textAlign': 'center', 'fontSize': '12px'})
    ])
])


# ─────────────────────────────────────────────────────────────
# 5. CALLBACK PRINCIPAL
# ─────────────────────────────────────────────────────────────
@app.callback(
    [
        Output('kpi-cards',              'children'),
        Output('chart-heatmap-hora-sev', 'figure'),
        Output('chart-dia-luz',          'figure'),
        Output('chart-top-states',       'figure'),
        Output('chart-top-cities',       'figure'),
        Output('chart-clima-count',      'figure'),
        Output('chart-clima-sev-prom',   'figure'),
        Output('chart-infraestructura',  'figure'),
        Output('chart-dur-severidad',    'figure'),
        Output('chart-dur-clima',        'figure'),
        Output('chart-dur-estado',       'figure'),
        Output('chart-severity-pie',     'figure'),
    ],
    [
        Input('filter-state',    'value'),
        Input('filter-severity', 'value'),
        Input('filter-clima',    'value'),
        Input('filter-twilight', 'value'),
        Input('filter-anio',     'value'),
    ]
)
def update_all(state, severity, clima, twilight, anio):

    # ── Filtrado ────────────────────────────────────────────
    df = df_main.copy()
    if state    != 'all': df = df[df['state']             == state]
    if severity != 'all': df = df[df['severity']          == int(severity)]
    if clima    != 'all': df = df[df['weather_condition'] == clima]
    if twilight != 'all': df = df[df['civil_twilight']    == twilight]
    if anio     != 'all': df = df[df['anio']              == int(anio)]

    total     = len(df)
    sev_alta  = int((df['severity'] >= 3).sum())
    tasa_alta = round(sev_alta / total * 100, 1) if total > 0 else 0.0
    dur_prom  = round(df['duracion_minutos'].mean(), 1) if total > 0 else 0.0
    dist_prom = round(df['distance'].mean(), 2) if total > 0 else 0.0

    # ── KPIs ────────────────────────────────────────────────
    kpi_defs = [
        ("Tasa Sev. Alta (3-4)", f"{tasa_alta} %",   '#9b59b6'),
        ("Duración Promedio",    f"{dur_prom} min",   '#3498db'),
        ("Total Accidentes",     f"{total:,}",         '#e74c3c'),
        ("Accidentes Sev. 3-4", f"{sev_alta:,}",      '#e67e22'),
        ("Distancia Prom.",      f"{dist_prom} mi",   '#27ae60'),
    ]
    kpi_cards = [
        html.Div([
            html.H3(v, style={'color': c, 'margin': '0', 'fontSize': '22px'}),
            html.P(l, style={'color': '#7f8c8d', 'margin': '0', 'fontSize': '12px'}),
        ], style=CARD)
        for l, v, c in kpi_defs
    ]

    # ── Heatmap hora × severidad ────────────────────────────
    hora_sev = (df.groupby(['hora', 'severity'])
                  .size()
                  .reset_index(name='count'))
    pivot = (hora_sev.pivot(index='severity', columns='hora', values='count')
                     .fillna(0)
                     .reindex(sorted(hora_sev['severity'].unique())))

    fig_heatmap = go.Figure(go.Heatmap(
        z=pivot.values,
        x=[f"{int(h):02d}:00" for h in pivot.columns],
        y=[f"Sev {int(s)}" for s in pivot.index],
        colorscale='YlOrRd',
        colorbar=dict(title='N° accidentes'),
    ))
    fig_heatmap.update_layout(
        title='Accidentes por Hora del Día y Severidad<br>'
              '<sup>¿En qué horarios ocurren más accidentes graves?</sup>',
        xaxis_title='Hora del día',
        yaxis_title='Severidad',
        margin=dict(t=70, b=50),
    )

    # ── Día semana × condición de luz ───────────────────────
    # FIX: dia_semana es texto en inglés ('Monday', etc.)
    dia_luz = (df.groupby(['dia_semana', 'civil_twilight'])
                 .size()
                 .reset_index(name='count'))
    dia_luz['dia_nombre'] = dia_luz['dia_semana'].map(DIAS_MAP_EN).fillna(dia_luz['dia_semana'])

    fig_dia_luz = px.bar(
        dia_luz, x='dia_nombre', y='count', color='civil_twilight',
        color_discrete_map={'Day': '#f39c12', 'Night': '#2c3e50'},
        category_orders={'dia_nombre': DIAS_ORDER},
        labels={'count': 'N° Accidentes', 'dia_nombre': 'Día', 'civil_twilight': 'Luz'},
        title='Accidentes por Día y Condición de Luz<br>'
              '<sup>Día vs Noche por día de la semana</sup>',
        barmode='stack',
    )
    fig_dia_luz.update_layout(margin=dict(t=70, b=40),
                              legend=dict(orientation='h', y=1.08))

    # ── Top 15 estados ──────────────────────────────────────
    states_df = (df.groupby('state')
                   .agg(
                       total    =('id', 'count'),
                       graves   =('severity', lambda x: (x >= 3).sum()),
                       sev_prom =('severity', 'mean'),
                   )
                   .reset_index()
                   .nlargest(15, 'total'))

    fig_states = px.bar(
        states_df, x='total', y='state', orientation='h',
        color='sev_prom', color_continuous_scale='Reds',
        text='graves',
        labels={'total': 'Total accidentes', 'state': 'Estado',
                'sev_prom': 'Sev. prom.', 'graves': 'Graves (3-4)'},
        title='Top 15 Estados — Total y Severidad<br>'
              '<sup>¿Qué estados concentran más accidentes graves?</sup>',
    )
    fig_states.update_traces(texttemplate='%{text:,}', textposition='outside')
    fig_states.update_layout(yaxis={'categoryorder': 'total ascending'},
                             margin=dict(t=70, b=30))

    # ── Top 15 ciudades ─────────────────────────────────────
    cities_df = (df.groupby('city')
                   .agg(
                       total    =('id', 'count'),
                       graves   =('severity', lambda x: (x >= 3).sum()),
                       sev_prom =('severity', 'mean'),
                   )
                   .reset_index()
                   .nlargest(15, 'graves'))

    fig_cities = px.bar(
        cities_df, x='graves', y='city', orientation='h',
        color='sev_prom', color_continuous_scale='Oranges',
        labels={'graves': 'Accidentes Sev. 3-4', 'city': 'Ciudad',
                'sev_prom': 'Sev. prom.'},
        title='Top 15 Ciudades — Accidentes de Alta Severidad<br>'
              '<sup>¿Qué ciudades concentran más accidentes graves?</sup>',
    )
    fig_cities.update_layout(yaxis={'categoryorder': 'total ascending'},
                             margin=dict(t=70, b=30))

    # ── Distribución accidentes por clima ───────────────────
    clima_count = (df.groupby('weather_condition')
                     .agg(total=('id', 'count'))
                     .reset_index()
                     .nlargest(15, 'total'))

    fig_clima_count = px.bar(
        clima_count, x='total', y='weather_condition', orientation='h',
        color='total', color_continuous_scale='Blues',
        labels={'total': 'N° Accidentes', 'weather_condition': 'Condición climática'},
        title='Distribución de Accidentes por Clima (Top 15)<br>'
              '<sup>¿Qué clima concentra más accidentes?</sup>',
    )
    fig_clima_count.update_layout(yaxis={'categoryorder': 'total ascending'},
                                  margin=dict(t=70, b=30), showlegend=False)

    # ── Severidad promedio por clima ─────────────────────────
    clima_sev = (df[df['severity'] >= 3]
                   .groupby('weather_condition')
                   .agg(
                       graves   =('id', 'count'),
                       sev_prom =('severity', 'mean'),
                   )
                   .reset_index()
                   .query('graves >= 5')
                   .nlargest(15, 'graves'))

    fig_clima_sev = px.bar(
        clima_sev, x='graves', y='weather_condition', orientation='h',
        color='sev_prom', color_continuous_scale='Reds',
        labels={'graves': 'Accidentes Sev. 3-4',
                'weather_condition': 'Condición climática',
                'sev_prom': 'Sev. prom.'},
        title='Clima vs Accidentes Graves (Sev. 3-4) — Top 15<br>'
              '<sup>¿Qué clima se asocia más a accidentes graves?</sup>',
    )
    fig_clima_sev.update_layout(yaxis={'categoryorder': 'total ascending'},
                                margin=dict(t=70, b=30))

    # ── Infraestructura crítica ──────────────────────────────
    # FIX: ahora las columnas son int 0/1 (convertidas en load_data)
    infra_cols = {
        'Cruce peatonal': 'crossing',
        'Intersección':   'junction',
        'Estación':       'station',
        'Señal Stop':     'stop',
        'Semáforo':       'traffic_signal',
    }
    rows = []
    for nombre, col in infra_cols.items():
        sub        = df[df[col] == 1]
        total_col  = len(sub)
        graves_col = int((sub['severity'] >= 3).sum())
        tasa_col   = round(graves_col / total_col * 100, 1) if total_col > 0 else 0.0
        rows.append({
            'infraestructura': nombre,
            'total':           total_col,
            'graves':          graves_col,
            'tasa_graves_%':   tasa_col,
        })
    infra_df = pd.DataFrame(rows)

    fig_infra = go.Figure()
    fig_infra.add_trace(go.Bar(
        name='Total accidentes', x=infra_df['infraestructura'], y=infra_df['total'],
        marker_color='#3498db', opacity=0.7,
        text=infra_df['total'].apply(lambda v: f"{v:,}"),
        textposition='outside',
    ))
    fig_infra.add_trace(go.Bar(
        name='Sev. alta (3-4)', x=infra_df['infraestructura'], y=infra_df['graves'],
        marker_color='#e74c3c', opacity=0.9,
        text=infra_df['tasa_graves_%'].apply(lambda v: f"{v}%"),
        textposition='outside',
    ))
    fig_infra.update_layout(
        title='Accidentes por Infraestructura Vial — Total vs Severidad Alta<br>'
              '<sup>Etiquetas rojas = % de accidentes graves sobre el total en esa infraestructura</sup>',
        barmode='group',
        xaxis_title='Tipo de infraestructura',
        yaxis_title='N° Accidentes',
        legend=dict(orientation='h', y=1.08),
        margin=dict(t=80, b=40),
    )

    # ── Duración prom. por severidad ────────────────────────
    dur_sev = (df.groupby('severity')
                 .agg(dur_prom=('duracion_minutos', 'mean'))
                 .reset_index()
                 .sort_values('severity'))
    dur_sev['sev_label'] = dur_sev['severity'].map(
        {1: 'Leve (1)', 2: 'Moderado (2)', 3: 'Grave (3)', 4: 'Muy Grave (4)'}
    )
    # FIX: color_discrete_map necesita claves del mismo tipo que la columna
    dur_sev['severity_int'] = dur_sev['severity'].astype(int)
    fig_dur_sev = px.bar(
        dur_sev, x='sev_label', y='dur_prom',
        color='severity_int',
        color_discrete_map=COLOR_SEV,
        text=dur_sev['dur_prom'].round(0).fillna(0).astype(int),
        labels={'dur_prom': 'Duración prom. (min)', 'sev_label': 'Severidad',
                'severity_int': 'Severidad'},
        title='Duración Prom. por Severidad<br>'
              '<sup>¿Cuánto dura según la gravedad?</sup>',
    )
    fig_dur_sev.update_traces(textposition='outside')
    fig_dur_sev.update_layout(showlegend=False, margin=dict(t=70, b=40))

    # ── Duración prom. por clima ─────────────────────────────
    dur_clima = (df.groupby('weather_condition')
                   .agg(dur_prom=('duracion_minutos', 'mean'),
                        total   =('id', 'count'))
                   .reset_index()
                   .query('total >= 10')
                   .nlargest(12, 'dur_prom'))
    fig_dur_clima = px.bar(
        dur_clima, x='dur_prom', y='weather_condition', orientation='h',
        color='dur_prom', color_continuous_scale='Blues',
        text=dur_clima['dur_prom'].round(0).fillna(0).astype(int),
        labels={'dur_prom': 'Duración prom. (min)',
                'weather_condition': 'Condición climática'},
        title='Duración Prom. por Clima (Top 12)<br>'
              '<sup>¿Qué clima prolonga más los accidentes?</sup>',
    )
    fig_dur_clima.update_traces(textposition='outside')
    fig_dur_clima.update_layout(yaxis={'categoryorder': 'total ascending'},
                                showlegend=False, margin=dict(t=70, b=30))

    # ── Duración prom. por estado ────────────────────────────
    dur_estado = (df.groupby('state')
                    .agg(dur_prom=('duracion_minutos', 'mean'),
                         total   =('id', 'count'))
                    .reset_index()
                    .query('total >= 10')
                    .nlargest(12, 'dur_prom'))
    fig_dur_estado = px.bar(
        dur_estado, x='dur_prom', y='state', orientation='h',
        color='dur_prom', color_continuous_scale='Purples',
        text=dur_estado['dur_prom'].round(0).fillna(0).astype(int),
        labels={'dur_prom': 'Duración prom. (min)', 'state': 'Estado'},
        title='Duración Prom. por Estado (Top 12)<br>'
              '<sup>¿En qué zonas duran más los accidentes?</sup>',
    )
    fig_dur_estado.update_traces(textposition='outside')
    fig_dur_estado.update_layout(yaxis={'categoryorder': 'total ascending'},
                                 showlegend=False, margin=dict(t=70, b=30))

    # ── Pie severidad ────────────────────────────────────────
    sev_pie = df.groupby('severity').size().reset_index(name='count')
    sev_pie['label'] = sev_pie['severity'].map(
        {1: 'Leve (1)', 2: 'Moderado (2)', 3: 'Grave (3)', 4: 'Muy Grave (4)'}
    )
    fig_pie = px.pie(
        sev_pie, values='count', names='label',
        color='severity',
        color_discrete_map={1: '#2ecc71', 2: '#f39c12', 3: '#e67e22', 4: '#e74c3c'},
        title='Distribución General por Nivel de Severidad',
        hole=0.35,
    )
    fig_pie.update_traces(textposition='inside', textinfo='percent+label')
    fig_pie.update_layout(showlegend=True, margin=dict(t=60, b=20))

    return (
        kpi_cards,
        fig_heatmap, fig_dia_luz,
        fig_states,  fig_cities,
        fig_clima_count, fig_clima_sev,
        fig_infra,
        fig_dur_sev, fig_dur_clima, fig_dur_estado,
        fig_pie,
    )


# ─────────────────────────────────────────────────────────────
# 6. ENTRY POINT
# ─────────────────────────────────────────────────────────────
if __name__ == '__main__':
    print("Dashboard disponible en: http://localhost:8050")
    app.run(debug=True, port=8050)

ModuleNotFoundError: No module named 'dash'